# Exploratory Analysis — SGCC Electricity Theft Dataset

Loads the real raw CSV via `src.data.loader` and the processed artifacts produced by `scripts/prepare_data.py`, and reproduces the key dataset-quality numbers shown in the dashboard's Data Exploration page. Run `python scripts/prepare_data.py` before this notebook.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, yaml

ROOT = Path.cwd().parent if (Path.cwd() / 'notebooks').exists() is False else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.loader import load_raw_data
from src.data.validation import validate_raw_dataset, format_report

with open(ROOT / 'configs' / 'config.yaml') as f:
    config = yaml.safe_load(f)

df, schema = load_raw_data(str(ROOT / config['paths']['raw_data']), config)
print(f"Loaded {len(df)} customers x {len(schema.date_cols_sorted)} dates")

In [ ]:
report = validate_raw_dataset(df, schema)
print(format_report(report))

In [ ]:
# Processed engineered features (run scripts/prepare_data.py first)
processed_dir = ROOT / config['paths']['processed_dir']
features = np.load(processed_dir / 'features_raw.npy')
labels = np.load(processed_dir / 'labels.npy')
import json
feature_names = json.load(open(processed_dir / 'feature_names.json'))
pd.DataFrame(features, columns=feature_names).assign(FLAG=labels).groupby('FLAG').mean().T